In [1]:
import os
import random
import sys
import numpy as np
import pandas as pd



In [2]:
SEED = 31
TARGET = "scalar_coupling_constant"
PREDICTION = "pred"


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(SEED)



In [3]:
train_path = "../input/champs-scalar-coupling/train.csv"
test_path = "../input/champs-scalar-coupling/test.csv"
structures_path = "../input/champs-scalar-coupling/structures.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
structures_df = pd.read_csv(structures_path)

print(
    f"train shape: {train_df.shape}, test shape: {test_df.shape}, structures shape: {structures_df.shape}"
)


def group_mean_log_mae(y_true, y_pred, types, floor=1e-9):
    """
    Fast metric computation for this competition:
    log(mean absolute error) per coupling type, then averaged.
    """
    maes = (y_true - y_pred).abs().groupby(types).mean()
    maes = np.log(maes.map(lambda x: max(x, floor)))
    return maes.mean()


atom0 = structures_df.rename(
    columns={
        "atom_index": "atom_index_0",
        "atom": "atom_0",
        "x": "x0",
        "y": "y0",
        "z": "z0",
    }
)[["molecule_name", "atom_index_0", "atom_0", "x0", "y0", "z0"]]

atom1 = structures_df.rename(
    columns={
        "atom_index": "atom_index_1",
        "atom": "atom_1",
        "x": "x1",
        "y": "y1",
        "z": "z1",
    }
)[["molecule_name", "atom_index_1", "atom_1", "x1", "y1", "z1"]]


def enrich(df):
    df = df.merge(atom0, on=["molecule_name", "atom_index_0"], how="left")
    df = df.merge(atom1, on=["molecule_name", "atom_index_1"], how="left")
    df["dist"] = np.sqrt(
        (df["x0"] - df["x1"]) ** 2
        + (df["y0"] - df["y1"]) ** 2
        + (df["z0"] - df["z1"]) ** 2
    )
    df["type_atom_pair"] = df["type"] + "_" + df["atom_0"] + "_" + df["atom_1"]
    df["dist_bin"] = pd.qcut(df["dist"], q=10, duplicates="drop")
    df["type_dist_bin"] = df["type"].astype(str) + "_" + df["dist_bin"].astype(str)
    return df


train_enriched = enrich(train_df)
test_enriched = enrich(test_df)

key_means = train_enriched.groupby("type_atom_pair")[TARGET].mean()
type_means = train_enriched.groupby("type")[TARGET].mean()
global_mean = train_enriched[TARGET].mean()
dist_means = train_enriched.groupby("type_dist_bin")[TARGET].mean()


def compute_group_stats(g):
    if len(g) < 2:
        slope = 0.0
    else:
        cov = np.cov(g["dist"], g[TARGET], bias=True)[0, 1]
        var = np.var(g["dist"])
        slope = cov / (var + 1e-9)
        slope = np.clip(slope, -0.5, 0.5)
    return pd.Series(
        {
            "mean_target": g[TARGET].mean(),
            "mean_dist": g["dist"].mean(),
            "slope": slope,
        }
    )


group_stats = train_enriched.groupby("type_atom_pair").apply(compute_group_stats)

mean_target_map = group_stats["mean_target"]
mean_dist_map = group_stats["mean_dist"]
slope_map = group_stats["slope"]

bin_stats = train_enriched.groupby("type_dist_bin").apply(compute_group_stats)

bin_mean_target = bin_stats["mean_target"]
bin_mean_dist = bin_stats["mean_dist"]
bin_slope = bin_stats["slope"]

shuffled = train_enriched.sample(frac=1, random_state=SEED)
split_idx = int(0.8 * len(shuffled))
val_df = shuffled.iloc[split_idx:]




train shape: (4191263, 6), test shape: (467813, 5), structures shape: (1379964, 6)


/tmp/ipykernel_11/939170225.py:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_stats = train_enriched.groupby("type_atom_pair").apply(compute_group_stats)
/tmp/ipykernel_11/939170225.py:91: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bin_stats = train_enriched.groupby("type_dist_bin").apply(compute_group_stats)


In [4]:
def predict_series(df):
    """
    Predict using the mean‑+‑slope baseline.
    Alpha is set to 0.3 (more weight on the robust type mean) to
    reduce over‑fitting and move the log‑MAE closer to the target.
    Predictions are clipped to non‑negative values because coupling
    constants cannot be negative.
    """
    alpha = 0.3  # give more weight to the stable type‑level mean

    type_mean_series = df["type"].map(type_means)

    forward_pair = df["type_atom_pair"].map(mean_target_map)
    forward_adj = (df["dist"] - df["type_atom_pair"].map(mean_dist_map).fillna(0)) * df[
        "type_atom_pair"
    ].map(slope_map).fillna(0)
    forward = forward_pair + forward_adj
    forward = forward * alpha + type_mean_series * (1 - alpha)

    rev_key = df["type"] + "_" + df["atom_1"] + "_" + df["atom_0"]
    rev_pair = rev_key.map(mean_target_map)
    rev_adj = (df["dist"] - rev_key.map(mean_dist_map).fillna(0)) * rev_key.map(
        slope_map
    ).fillna(0)
    rev = rev_pair + rev_adj
    rev = rev * alpha + type_mean_series * (1 - alpha)

    combined = forward.copy()
    both_exist = forward.notna() & rev.notna()
    combined[both_exist] = (forward[both_exist] + rev[both_exist]) / 2.0
    only_rev = combined.isna() & rev.notna()
    combined[only_rev] = rev[only_rev]

    fb1 = df["type_dist_bin"].map(dist_means)  # raw bin mean
    fb1_adj = fb1 + (
        df["dist"] - df["type_dist_bin"].map(bin_mean_dist).fillna(0)
    ) * df["type_dist_bin"].map(bin_slope).fillna(0)

    fb2 = type_mean_series

    combined = combined.fillna(fb1_adj)
    combined = combined.fillna(fb2)
    combined = combined.fillna(global_mean)

    combined = combined.clip(lower=0)

    return combined


val_pred = predict_series(val_df)

baseline_score = group_mean_log_mae(val_df[TARGET], val_pred, val_df["type"])
print(f"Improved validation score (log‑MAE): {baseline_score:.5f}")



Improved validation score (log‑MAE): 1.40905


In [5]:
test_pred = predict_series(test_enriched)

submission = test_enriched[["id"]].copy()
submission[TARGET] = test_pred

print("First few predictions:")
print(submission.head())



First few predictions:
        id  scalar_coupling_constant
0  2324604                 94.971200
1  2324605                  0.000000
2  2324606                  3.689004
3  2324607                  3.689004
4  2324608                  0.000000


In [6]:
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")
print("Current directory contents:", os.listdir("."))

Submission written to submission.csv
Current directory contents: ['champs-scalar-coupling_ruhong_champs-scalar-coupling-blend_v1_C1.ipynb', 'champs-scalar-coupling', 'submission.csv']
